# 0.3.1 - CatBoost Evaluation

Test-set evaluation: regression on the log scale and binary classification at the reference level.

## Setup

In [ ]:
import pickle

import catboost as cb
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report

from evaluation import (
    bootstrap_ci,
    load_radon_map,
    load_test_egid,
    load_test_set,
    map_matched_threshold,
    plot_calibrated_predicted_vs_observed,
    operating_points,
    permutation_importance,
    plot_confusion_matrix,
    plot_feature_importance,
    plot_precision_recall,
    plot_predicted_vs_observed,
    regression_scores,
    test_scores,
    threshold_for_recall,
)

ROOT = '../../'
DIR = '../../models/'

## Data and model

In [ ]:
X_test, y_test = load_test_set(ROOT)
egid_test = load_test_egid(ROOT)
with open(f'{DIR}0.2-encoding-pipeline.pkl', 'rb') as file:
    preprocessor = pickle.load(file)

model = cb.CatBoostRegressor()
model.load_model(f'{DIR}0.3-catboost-log-model.cbm')
y_pred = np.expm1(model.predict(preprocessor.transform(X_test)))

val = pd.read_csv(f'{DIR}0.3-catboost-val-predictions.csv', index_col=0)

## Regression (log scale)

In [ ]:
plot_predicted_vs_observed(y_test, y_pred, 'Predicted vs observed, test subset - CatBoost')
regression_scores(y_test, y_pred)

## Classification

Back-transformed predictions approximate the median concentration, so the decision threshold is chosen on the validation predictions: the model must detect as many exceedances as the FOPH radon map flagging probabilities above 1 % (the boundary of its lowest legend class). Thresholds are calibrated on measurements, because atmos predicts for one room at a time. The comparison with the radon map is in 0.5.3, on the quantity the map describes. Change the settings below and re-run from there.

In [ ]:
RADON_REFERENCE_LEVEL = 300  # Bq/m3
OPERATING_RECALLS = np.round(np.arange(0.50, 0.951, 0.05), 2)
RADON_MAP_CUTOFF = 0.01  # FOPH legend: lowest class ≤ 1 %

val['radon_map'] = load_radon_map(ROOT, val.index)
DECISION_RECALL, DECISION_THRESHOLD = map_matched_threshold(val, RADON_MAP_CUTOFF, RADON_REFERENCE_LEVEL)

val_exceeds = val['observed'] >= RADON_REFERENCE_LEVEL
thresholds = {f'Recall ≥ {r:.0%}': threshold_for_recall(val_exceeds, val['predicted'], r) for r in OPERATING_RECALLS}
thresholds[f'Decision (map > {RADON_MAP_CUTOFF:.0%})'] = DECISION_THRESHOLD

print(f'Decision threshold: {DECISION_THRESHOLD:.0f} Bq/m3 (recall of the radon map > {RADON_MAP_CUTOFF:.0%} on validation: {DECISION_RECALL:.1%})')
operating_points(val_exceeds, val['predicted'], thresholds)

In [ ]:
# Test subset, thresholds from the validation subset
test_exceeds = y_test >= RADON_REFERENCE_LEVEL
operating_points(test_exceeds, y_pred, thresholds)

In [ ]:
plot_precision_recall(test_exceeds, y_pred, 'Precision–recall, test subset - CatBoost', path=f'{ROOT}reports/figure-precision-recall-catboost')

In [ ]:
flagged = y_pred >= DECISION_THRESHOLD
labels = [f'< {RADON_REFERENCE_LEVEL}', f'≥ {RADON_REFERENCE_LEVEL}']
print(classification_report(test_exceeds, flagged, target_names=labels))
plot_confusion_matrix(test_exceeds, flagged, f'Decision threshold {DECISION_THRESHOLD:.0f} Bq/m3 - CatBoost', RADON_REFERENCE_LEVEL)

In [ ]:
plot_calibrated_predicted_vs_observed(y_test, y_pred, DECISION_THRESHOLD, RADON_REFERENCE_LEVEL, 'Calibrated predictions, test subset - CatBoost')

### Confidence intervals

Bootstrap on the test subset, resampling buildings (1,000 samples).

In [ ]:
bootstrap_ci(lambda t, p: test_scores(t, p, DECISION_THRESHOLD, RADON_REFERENCE_LEVEL), y_test, y_pred, egid_test)

## Feature importance

Permutation importance on the original features, so that one-hot dummies and missing-value indicators count with their feature.

In [ ]:
# Drop in log-scale R² when one original feature (before encoding) is shuffled
importance = permutation_importance(lambda X: np.expm1(model.predict(preprocessor.transform(X))), X_test, y_test)
plot_feature_importance(importance.index, importance.values, 'Permutation importance, test subset - CatBoost', xlabel='Decrease in R² (log)')